# US CPI and Energy Producer Prices, 1957-2022

This notebook loads `fred_data.csv` and visualizes the relationship between the US Consumer Price Index (CPI) and energy producer prices (FPP). The first chart compares their log levels over time, while the second chart shows how the correlation of their monthly log growth rates changes through rolling 36-month windows.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.patches import Patch

if "seaborn-v0_8-whitegrid" in plt.style.available:
    plt.style.use("seaborn-v0_8-whitegrid")
else:
    plt.style.use("default")

plt.rcParams.update({
    "axes.labelsize": 12,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 10,
    "axes.titlesize": 14,
})

BLUE = "#1f77b4"
ORANGE = "#ff7f0e"
SHOCK_SHADE = "#d9d9d9"
STRUCTURAL_BREAK = pd.Timestamp("2000-01-01")

shock_periods = [
    ("1973-10-01", "1975-03-01", "Oil embargo"),
    ("1979-01-01", "1980-07-01", "Iranian Revolution"),
    ("1990-07-01", "1991-03-01", "Gulf War"),
    ("2007-12-01", "2009-06-01", "GFC"),
    ("2020-02-01", "2020-04-01", "COVID"),
]

def minimize_spines(*axes):
    for axis in axes:
        axis.spines["top"].set_visible(False)
        axis.spines["right"].set_alpha(0.35)
        axis.spines["left"].set_alpha(0.35)
        axis.spines["bottom"].set_alpha(0.35)
        axis.grid(False)

def format_time_axis(axis):
    axis.xaxis.set_major_locator(mdates.YearLocator(base=10))
    axis.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    axis.tick_params(axis="both", labelsize=10)


In [ ]:
data = pd.read_csv("fred_data.csv", parse_dates=["date"]).set_index("date").sort_index()

required_columns = {"CPI", "CPI_core", "FPP"}
missing_columns = required_columns.difference(data.columns)
if missing_columns:
    raise ValueError(f"Missing required columns: {sorted(missing_columns)}")

analysis = data.loc["1957-01-01":"2022-05-01", ["CPI", "FPP"]].copy()
if analysis.empty:
    raise ValueError("No observations available for the requested analysis period.")

analysis.head()


## Chart 1 - CPI and Energy Producer Price Log Levels

This chart compares the log level of US CPI with the log level of energy producer prices. The shaded regions mark major oil, financial, and pandemic shocks, making it easier to see when energy-price disruptions coincided with broader consumer-price movements. The dashed vertical line marks January 2000 as a structural break reference point.


In [ ]:
log_levels = np.log(analysis[["CPI", "FPP"]])

fig, ax_cpi = plt.subplots(figsize=(14, 5))
ax_fpp = ax_cpi.twinx()

line_cpi, = ax_cpi.plot(
    log_levels.index,
    log_levels["CPI"],
    color=BLUE,
    linewidth=2.2,
    label="log CPI",
)
line_fpp, = ax_fpp.plot(
    log_levels.index,
    log_levels["FPP"],
    color=ORANGE,
    linewidth=2.2,
    label="log FPP",
)

for start, end, label in shock_periods:
    start_date = pd.Timestamp(start)
    end_date = pd.Timestamp(end)
    ax_cpi.axvspan(start_date, end_date, color=SHOCK_SHADE, alpha=0.45, zorder=0)
    midpoint = start_date + (end_date - start_date) / 2
    ax_cpi.text(
        midpoint,
        0.98,
        label,
        transform=ax_cpi.get_xaxis_transform(),
        ha="center",
        va="top",
        rotation=90,
        fontsize=9,
        color="#555555",
    )

ax_cpi.axvline(STRUCTURAL_BREAK, color="#333333", linestyle="--", linewidth=1.4)
ax_cpi.text(
    STRUCTURAL_BREAK,
    0.08,
    "Structural break",
    transform=ax_cpi.get_xaxis_transform(),
    ha="left",
    va="bottom",
    rotation=90,
    fontsize=10,
    color="#333333",
)

ax_cpi.set_title("US CPI and Energy Producer Prices (log levels), 1957–2022", pad=12)
ax_cpi.set_ylabel("log CPI", color=BLUE, fontsize=12)
ax_fpp.set_ylabel("log FPP", color=ORANGE, fontsize=12)
ax_cpi.tick_params(axis="y", labelcolor=BLUE, labelsize=10)
ax_fpp.tick_params(axis="y", labelcolor=ORANGE, labelsize=10)
format_time_axis(ax_cpi)
minimize_spines(ax_cpi, ax_fpp)

ax_cpi.legend(
    handles=[line_cpi, line_fpp],
    loc="upper left",
    frameon=False,
)
fig.tight_layout()
fig.savefig("chart1_levels.png", dpi=300, bbox_inches="tight")
plt.show()


## Chart 2 - Rolling Correlation of CPI and Energy Price Growth

This chart shows the 36-month rolling Pearson correlation between monthly CPI growth and monthly energy producer price growth, both measured as changes in log values. Green shading highlights periods of high co-movement above 0.7, while red shading highlights periods of low co-movement below 0.3. The dashed horizontal line shows the full-sample benchmark correlation of 0.56.


In [ ]:
log_growth = np.log(analysis[["CPI", "FPP"]]).diff().dropna()
rolling_corr = log_growth["CPI"].rolling(window=36).corr(log_growth["FPP"])
full_sample_corr = 0.56

fig, ax = plt.subplots(figsize=(14, 5))
ax.axhspan(0.7, 1.0, color="#c7e9c0", alpha=0.45, label="High co-movement (>0.7)")
ax.axhspan(-1.0, 0.3, color="#fcbba1", alpha=0.35, label="Low co-movement (<0.3)")
line_corr, = ax.plot(
    rolling_corr.index,
    rolling_corr,
    color="#2c3e50",
    linewidth=2.2,
    label="36-month rolling correlation",
)
hline = ax.axhline(
    full_sample_corr,
    color="#555555",
    linestyle="--",
    linewidth=1.4,
    label="Full-sample correlation (0.56)",
)
ax.axvline(STRUCTURAL_BREAK, color="#333333", linestyle="--", linewidth=1.4)
ax.text(
    STRUCTURAL_BREAK,
    -0.92,
    "Structural break",
    ha="left",
    va="bottom",
    rotation=90,
    fontsize=10,
    color="#333333",
)

ax.set_title("Rolling 36-Month Correlation: CPI Growth vs Energy Price Growth", pad=12)
ax.set_ylabel("Pearson correlation", fontsize=12)
ax.set_ylim(-1.0, 1.0)
format_time_axis(ax)
minimize_spines(ax)
ax.legend(loc="lower left", frameon=False, ncol=2)
fig.tight_layout()
fig.savefig("chart2_rolling_corr.png", dpi=300, bbox_inches="tight")
plt.show()
